In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/equal-neutral-submit-version/train_equal.csv
/kaggle/input/equal-neutral-submit-version/test_equal.csv


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import ShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import re

# I fixed seed as If someone want to rerun my code, the result should be the same or very close
torch.manual_seed(42)
np.random.seed(42)

2024-08-21 16:34:50.067391: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-21 16:34:50.067492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-21 16:34:50.201570: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
#Deleting URL and turn text into lowercase
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = text.lower()
    return text

In [4]:
# I will balance dataset based on the minority class
def balance_data(df, samples_per_class = None):
    if samples_per_class is None:
        samples_per_class = df['sentiment_equal'].value_counts().min()
    balance_data = []
    for sentiment in df['sentiment_equal'].unique():
        class_data = df[df['sentiment_equal'] ==sentiment]
        balance_data.append(class_data.sample(samples_per_class, random_state=42))
    
    balance_df = pd.concat(balance_data, ignore_index=True)
    return balance_df

In [5]:
# Computing Metric
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [6]:
def train_and_evaluate(train_dataset, test_dataset, num_epochs, best_params):
    model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)
    
    training_args = TrainingArguments(
        output_dir=f'./roberta_results_{num_epochs}_epochs',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=best_params['batch_size'],
        per_device_eval_batch_size=best_params['batch_size'],
        weight_decay=best_params['weight_decay'],
        learning_rate=best_params['learning_rate'],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    return trainer.evaluate(), trainer

In [7]:
def tokenize_function(data: Dict[str, List], tokenizer: RobertaTokenizer, max_length):
    return tokenizer(data['comment'], padding='max_length', truncation=True, max_length=max_length)

In [8]:
train_df = pd.read_csv('/kaggle/input/equal-neutral-submit-version/train_equal.csv')
test_df = pd.read_csv('/kaggle/input/equal-neutral-submit-version/test_equal.csv')
for df in [train_df, test_df]:
    df['comment'] = df['comment'].apply(clean_text)
sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
train_df['label'] = train_df['sentiment_equal'].map(sentiment_mapping)
test_df['label'] = test_df['sentiment_equal'].map(sentiment_mapping)

undersampled_train_df = balance_data(train_df)
undersampled_test_df = balance_data(test_df)


In [9]:
print(undersampled_train_df['sentiment_equal'].value_counts())
print(undersampled_test_df['sentiment_equal'].value_counts())

sentiment_equal
Positive    86702
Negative    86702
Neutral     86702
Name: count, dtype: int64
sentiment_equal
Positive    21378
Neutral     21378
Negative    21378
Name: count, dtype: int64


In [10]:
# The best hyperparameter set from the hyperparameter_tuning process
best_hyperparameters = {'learning_rate': 3.9793483981751164e-05, 'batch_size': 64, 'weight_decay': 7.10985806177796e-05, 'max_length': 128}

def tokenize_function(data: Dict[str, List], tokenizer: RobertaTokenizer, max_length):
    return tokenizer(data['comment'], padding='max_length', truncation=True, max_length=max_length)

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
tokenized_train = Dataset.from_pandas(undersampled_train_df).map( lambda x: tokenize_function(x, tokenizer, best_hyperparameters['max_length']), batched=True)
tokenized_test = Dataset.from_pandas(undersampled_test_df).map(
    lambda x: tokenize_function(x, tokenizer, best_hyperparameters['max_length']), batched=True)

best_accuracy = 0
best_trainer = None
best_epochs = 0

for epochs in [1, 2, 3]:
    print(f"\nTraining for {epochs} epochs:")
    metrics, trainer = train_and_evaluate(tokenized_train, tokenized_test, epochs, best_hyperparameters)

    print(f"Results for {epochs} epochs:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

    if metrics['eval_accuracy'] > best_accuracy:
        best_accuracy = metrics['eval_accuracy']
        best_trainer = trainer
        best_epochs = epochs

print(f"\nBest model achieved with {best_epochs} epochs")
best_trainer.save_model('./best_roberta_model')
print("Best model saved in './best_roberta_model' directory")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Map:   0%|          | 0/260106 [00:00<?, ? examples/s]

... (11 outputs left out)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.520600,0.521179,0.778822,0.775588,0.774671,0.778822
2,0.486400,0.508132,0.781302,0.777487,0.776810,0.781302
3,0.446700,0.516764,0.782346,0.779914,0.778858,0.782346


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

Results for 3 epochs:
eval_loss: 0.5168
eval_accuracy: 0.7823
eval_f1: 0.7799
eval_precision: 0.7789
eval_recall: 0.7823
eval_runtime: 244.7528
eval_samples_per_second: 262.0360
eval_steps_per_second: 2.0510
epoch: 3.0000

Best model achieved with 2 epochs
Best model saved in './best_roberta_model' directory
